# Compare VLMs — Side-by-Side

**HFA AI Training — Module 6: Vision Language Models**

This notebook sends the SAME image to multiple VLMs and compares their outputs.
It helps you understand the strengths and characteristics of each model
so you know when to use which one.

**Quick Guide — When to use which VLM:**
- **Gemini (Recommended):** Video analysis, massive image sets, spatial reasoning, speed, document parsing
- **GPT-4o:** General vision, creative descriptions, image generation

This notebook compares Gemini and GPT-4o side by side on the same image.

**Get keys at:**
- Google: https://aistudio.google.com/apikey
- OpenAI: https://platform.openai.com/api-keys

## Install Dependencies

In [ ]:
!pip install google-generativeai openai httpx

## Imports and Configuration

Load the required libraries and configure API keys. The script checks which
API keys are available and only tests models you have keys for.

In [ ]:
import os
import json
import time
import base64
import httpx

import google.generativeai as genai
from openai import OpenAI

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# Check which APIs are available.
available_models = []

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    available_models.append("gemini")
else:
    print("WARNING: GOOGLE_API_KEY not set — Gemini will be skipped.")

if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    available_models.append("gpt-4o")
else:
    print("WARNING: OPENAI_API_KEY not set — GPT-4o will be skipped.")

if not available_models:
    raise EnvironmentError(
        "No API keys found. Set at least one of:\n"
        "  export GOOGLE_API_KEY='your-key'\n"
        "  export OPENAI_API_KEY='your-key'"
    )

# Model versions to use.
GEMINI_MODEL = "gemini-2.0-flash"
GPT4O_MODEL = "gpt-4o"

print(f"Available models: {', '.join(available_models)}")

## Individual Model Calls

Helper functions that send an image and prompt to each model and return
the response along with timing information.

In [ ]:
def ask_gemini(image_bytes: bytes, prompt: str) -> dict:
    """
    Send an image and prompt to Google Gemini.

    Returns a dict with the response text and timing.
    """
    model = genai.GenerativeModel(GEMINI_MODEL)

    start_time = time.time()
    response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            prompt,
        ]
    )
    elapsed = time.time() - start_time

    return {
        "model": f"Gemini ({GEMINI_MODEL})",
        "response": response.text,
        "time_seconds": round(elapsed, 2),
    }

In [ ]:
def ask_gpt4o(image_bytes: bytes, prompt: str) -> dict:
    """
    Send an image and prompt to OpenAI GPT-4o.

    GPT-4o accepts images as base64-encoded data URLs.

    Returns a dict with the response text and timing.
    """
    # Encode image bytes to base64 for the OpenAI API.
    base64_data = base64.b64encode(image_bytes).decode("utf-8")

    start_time = time.time()
    response = openai_client.chat.completions.create(
        model=GPT4O_MODEL,
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_data}",
                        },
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    )
    elapsed = time.time() - start_time

    return {
        "model": f"GPT-4o ({GPT4O_MODEL})",
        "response": response.choices[0].message.content,
        "time_seconds": round(elapsed, 2),
    }

## Test 1: General Image Description

Ask each model to describe the same image and compare their styles.
This reveals how each model approaches visual description.

In [ ]:
def compare_general_description(image_bytes: bytes) -> None:
    """
    Test 1: General image description.

    Ask each model to describe the same image and compare their styles.
    This reveals how each model approaches visual description.
    """
    print("\n" + "=" * 70)
    print("  TEST 1: GENERAL IMAGE DESCRIPTION")
    print("  Prompt: 'Describe this image in detail.'")
    print("=" * 70)

    prompt = "Describe this image in detail. What do you see?"

    results = []

    if "gemini" in available_models:
        print(f"\n  Asking Gemini...")
        result = ask_gemini(image_bytes, prompt)
        results.append(result)
        print(f"  Gemini responded in {result['time_seconds']}s")

    if "gpt-4o" in available_models:
        print(f"\n  Asking GPT-4o...")
        result = ask_gpt4o(image_bytes, prompt)
        results.append(result)
        print(f"  GPT-4o responded in {result['time_seconds']}s")

    # Display results side by side.
    print("\n" + "-" * 70)
    for result in results:
        print(f"\n  {result['model']} ({result['time_seconds']}s):")
        print(f"  {'~' * 50}")
        # Indent and truncate for readability.
        text = result["response"][:500]
        for line in text.split("\n"):
            print(f"    {line}")
        if len(result["response"]) > 500:
            print(f"    ... ({len(result['response'])} total characters)")
    print()

## Test 2: Structured Data Extraction

Ask each model to extract JSON data from the same image.
This tests each model's ability to produce clean, structured output.

In [ ]:
def compare_structured_extraction(image_bytes: bytes) -> None:
    """
    Test 2: Structured data extraction.

    Ask each model to extract JSON data from the same image.
    This tests each model's ability to produce clean, structured output.
    """
    print("\n" + "=" * 70)
    print("  TEST 2: STRUCTURED DATA EXTRACTION")
    print("  Prompt: Extract property details as JSON")
    print("=" * 70)

    prompt = """Analyze this image and extract information into this exact JSON structure:
    {
        "property_type": "type of property",
        "architectural_style": "style description",
        "condition": "excellent/good/fair/poor",
        "key_features": ["feature1", "feature2", "feature3"],
        "exterior_materials": ["material1", "material2"],
        "estimated_era": "approximate construction period",
        "overall_impression": "one sentence summary"
    }

    Return ONLY valid JSON — no extra text or markdown."""

    results = []

    if "gemini" in available_models:
        print(f"\n  Asking Gemini...")
        result = ask_gemini(image_bytes, prompt)
        results.append(result)
        print(f"  Gemini responded in {result['time_seconds']}s")

    if "gpt-4o" in available_models:
        print(f"\n  Asking GPT-4o...")
        result = ask_gpt4o(image_bytes, prompt)
        results.append(result)
        print(f"  GPT-4o responded in {result['time_seconds']}s")

    # Parse and display results.
    print("\n" + "-" * 70)
    for result in results:
        print(f"\n  {result['model']} ({result['time_seconds']}s):")
        print(f"  {'~' * 50}")

        raw = result["response"].strip()
        # Strip markdown code blocks.
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1]
            raw = raw.rsplit("```", 1)[0]

        try:
            parsed = json.loads(raw)
            print(f"    Valid JSON: YES")
            print(f"    {json.dumps(parsed, indent=6)}")
        except json.JSONDecodeError:
            print(f"    Valid JSON: NO")
            print(f"    Raw response: {raw[:300]}")
    print()

## Test 3: Specific Analytical Question

Ask each model a specific question that requires reasoning about the image.
This tests depth of analysis and practical usefulness.

In [ ]:
def compare_specific_question(image_bytes: bytes) -> None:
    """
    Test 3: Specific analytical question.

    Ask each model a specific question that requires reasoning about the image.
    This tests depth of analysis and practical usefulness.
    """
    print("\n" + "=" * 70)
    print("  TEST 3: SPECIFIC ANALYTICAL QUESTION")
    print("  Prompt: Real estate listing analysis")
    print("=" * 70)

    prompt = """You are a real estate expert. Look at this property and answer:

    1. What are the top 3 selling points for this property?
    2. What potential concerns might a buyer have?
    3. What price range would you estimate (in USD) and why?
    4. What type of buyer would be most interested in this property?

    Be specific and practical in your analysis."""

    results = []

    if "gemini" in available_models:
        print(f"\n  Asking Gemini...")
        result = ask_gemini(image_bytes, prompt)
        results.append(result)
        print(f"  Gemini responded in {result['time_seconds']}s")

    if "gpt-4o" in available_models:
        print(f"\n  Asking GPT-4o...")
        result = ask_gpt4o(image_bytes, prompt)
        results.append(result)
        print(f"  GPT-4o responded in {result['time_seconds']}s")

    # Display results.
    print("\n" + "-" * 70)
    for result in results:
        print(f"\n  {result['model']} ({result['time_seconds']}s):")
        print(f"  {'~' * 50}")
        text = result["response"][:800]
        for line in text.split("\n"):
            print(f"    {line}")
        if len(result["response"]) > 800:
            print(f"    ... ({len(result['response'])} total characters)")
    print()

## Comparison Summary

A quick reference guide for when to use each VLM.

In [ ]:
def print_comparison_summary() -> None:
    """
    Print a summary of when to use each VLM.
    """
    print("\n" + "=" * 70)
    print("  COMPARISON SUMMARY: WHEN TO USE WHICH VLM")
    print("=" * 70)
    print("""
    GOOGLE GEMINI (Recommended) — Best for:
      - Video analysis and understanding (best in class)
      - Processing many images at once (huge context window)
      - Fast responses at lower cost
      - Spatial reasoning (where things are in an image)
      - Audio + image + text together
      - Document parsing and structured data extraction
      - Careful, thorough image analysis

    GPT-4o — Best for:
      - General-purpose vision tasks
      - Creative descriptions and captions
      - When you also need image generation
      - Integration with OpenAI's broader ecosystem

    RULE OF THUMB:
      - Need to analyze video?            -> Gemini
      - Need to parse a document?          -> Gemini
      - Need careful, detailed analysis?   -> Gemini
      - Need fast + cheap at scale?        -> Gemini
      - Need creative descriptions?        -> GPT-4o
      - Need image generation too?         -> GPT-4o
      - Not sure?                          -> Start with Gemini, it's the most versatile
    """)

## Run All Comparisons

Download the test image and run all three comparison tests.

In [ ]:
print("=" * 70)
print("  VLM COMPARISON: SAME IMAGE, DIFFERENT MODELS")
print("  See how Gemini and GPT-4o analyze the same image differently")
print("=" * 70)
print(f"\n  Available models: {', '.join(available_models)}")

# Sample image — a public domain property photo.
image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "1/19/Weatherboard_house_in_Coorparoo%2C_Queensland.jpg/"
    "1280px-Weatherboard_house_in_Coorparoo%2C_Queensland.jpg"
)

# Download the image once (needed for both Gemini and GPT-4o).
print(f"\n  Downloading test image...")
response = httpx.get(image_url)
image_bytes = response.content
print(f"  Image downloaded ({len(image_bytes):,} bytes)")

### Test 1: General Description

In [ ]:
compare_general_description(image_bytes)

### Test 2: Structured Extraction

In [ ]:
compare_structured_extraction(image_bytes)

### Test 3: Specific Analytical Question

In [ ]:
compare_specific_question(image_bytes)

### Summary: When to Use Which VLM

In [ ]:
print_comparison_summary()

In [ ]:
print("=" * 70)
print("  All comparisons complete!")
print("=" * 70)